In [ ]:
!pip install -q -U transformers accelerate bitsandbytes
!pip install -q langchain langchain-community langgraph
!pip install -q mcp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 7.0 MB/s eta 0:00:00


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading Qwen model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("✅ Qwen model loaded successfully!")

Loading tokenizer...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Qwen model loaded successfully!


In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

def qwen_generate(messages, max_new_tokens=200):

    formatted_messages = []

    for message in messages:

        if isinstance(message, HumanMessage):
            role = "user"
        elif isinstance(message, AIMessage):
            role = "assistant"
        else:
            role = "user"

        formatted_messages.append({
            "role": role,
            "content": message.content
        })

    text = tokenizer.apply_chat_template(
        formatted_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip()

print("✅ Qwen generation function ready!")

✅ Qwen generation function ready!


In [ ]:
expenses = []

print("✅ Expense storage initialized!")

✅ Expense storage initialized!


In [ ]:
from langchain_core.tools import tool

@tool
def add_expense(amount: float, category: str, description: str = "N/A"):
    """
    Add an expense to the expense tracker.
    """

    expense = {
        "amount": amount,
        "category": category,
        "description": description
    }

    expenses.append(expense)

    return (
        f"Expense added successfully: "
        f"₹{amount} | {category} | {description}"
    )

In [ ]:
print(add_expense.invoke({
    "amount": 200,
    "category": "Travel",
    "description": "Bus fare"
}))

Expense added successfully: ₹200.0 | Travel | Bus fare


In [ ]:
# CELL 10 — DEFINE GET EXPENSES TOOL

from langchain_core.tools import tool

@tool
def get_expenses():
    """
    Get all recorded expenses.
    """

    if not expenses:
        return "No expenses found."

    return "\n".join(
        f"₹{e['amount']} | {e['category']} | {e['description']}"
        for e in expenses
    )

print("✅ get_expenses tool created!")

✅ get_expenses tool created!


In [ ]:
print(get_expenses.invoke({}))

₹200.0 | Travel | Bus fare


In [ ]:
# CELL 11 — TEST BOTH EXPENSE TOOLS

print("ADD EXPENSE TOOL:")
print(add_expense.name)

print("\nGET EXPENSES TOOL:")
print(get_expenses.name)

print("\nCURRENT EXPENSES:")
print(get_expenses.invoke({}))

ADD EXPENSE TOOL:
add_expense

GET EXPENSES TOOL:
get_expenses

CURRENT EXPENSES:
₹200.0 | Travel | Bus fare


In [ ]:
# CELL 12 — CREATE MCP SERVER FILE

server_code = '''
from mcp.server.fastmcp import FastMCP
import json
import os

mcp = FastMCP("Expense Tracker MCP Server")

DATA_FILE = "expenses.json"


def load_expenses():
    if not os.path.exists(DATA_FILE):
        return []

    with open(DATA_FILE, "r") as f:
        return json.load(f)


def save_expenses(expenses):
    with open(DATA_FILE, "w") as f:
        json.dump(expenses, f, indent=2)


@mcp.tool()
def add_expense(
    amount: float,
    category: str,
    description: str = "N/A"
) -> str:
    """Add an expense to the expense tracker."""

    expenses = load_expenses()

    expense = {
        "amount": amount,
        "category": category,
        "description": description
    }

    expenses.append(expense)
    save_expenses(expenses)

    return (
        f"Expense added successfully: "
        f"₹{amount} | {category} | {description}"
    )


@mcp.tool()
def get_expenses() -> str:
    """Get all recorded expenses."""

    expenses = load_expenses()

    if not expenses:
        return "No expenses found."

    return "\\n".join(
        f"₹{e['amount']} | "
        f"{e['category']} | "
        f"{e['description']}"
        for e in expenses
    )


if __name__ == "__main__":
    mcp.run(transport="streamable-http")
'''

with open("mcp_server.py", "w") as f:
    f.write(server_code)

print("✅ mcp_server.py created successfully!")

✅ mcp_server.py created successfully!


In [ ]:
# CELL 13 — SAVE EXISTING EXPENSES FOR MCP

import json

with open("expenses.json", "w") as f:
    json.dump(expenses, f, indent=2)

print("✅ Existing expenses transferred to MCP storage!")
print(expenses)

✅ Existing expenses transferred to MCP storage!
[{'amount': 200.0, 'category': 'Travel', 'description': 'Bus fare'}]


In [ ]:
# FIX MCP VERSION

!pip uninstall -y mcp fastmcp
!pip install -q "mcp==1.9.4"

print("✅ MCP 1.9.4 installed!")

Found existing installation: mcp 2.2.0
Uninstalling mcp-2.2.0:
  Successfully uninstalled mcp-2.2.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 10.3 MB/s eta 0:00:00
✅ MCP 1.9.4 installed!


In [ ]:
import mcp
import mcp.server.fastmcp

print("MCP version:", getattr(mcp, "__version__", "1.9.4"))
print("✅ FastMCP import successful!")

MCP version: 1.9.4
✅ FastMCP import successful!


In [ ]:
# CELL 4 — EXPENSE STORAGE

expenses = []

print("✅ Expense storage initialized!")

✅ Expense storage initialized!


In [ ]:
# Add our first sample expense

expenses.append({
    "amount": 200,
    "category": "Travel",
    "description": "Bus fare"
})

print("✅ Sample expense added!")
print(expenses)

✅ Sample expense added!
[{'amount': 200, 'category': 'Travel', 'description': 'Bus fare'}]


In [ ]:
# CELL 5 — ADD EXPENSE TOOL

from langchain_core.tools import tool

@tool
def add_expense(
    amount: float,
    category: str,
    description: str = "N/A"
):
    """
    Add an expense to the expense tracker.
    """

    expense = {
        "amount": amount,
        "category": category,
        "description": description
    }

    expenses.append(expense)

    return (
        f"Expense added successfully: "
        f"₹{amount} | {category} | {description}"
    )

print("✅ add_expense tool created!")

✅ add_expense tool created!


In [ ]:
print(add_expense.invoke({
    "amount": 150,
    "category": "Food",
    "description": "Lunch"
}))

Expense added successfully: ₹150.0 | Food | Lunch


In [ ]:
# CELL 6 — GET EXPENSES TOOL

@tool
def get_expenses():
    """
    Get all recorded expenses.
    """

    if not expenses:
        return "No expenses found."

    return "\n".join(
        f"₹{e['amount']} | {e['category']} | {e['description']}"
        for e in expenses
    )

print("✅ get_expenses tool created!")

✅ get_expenses tool created!


In [ ]:
print(get_expenses.invoke({}))

₹200 | Travel | Bus fare
₹150.0 | Food | Lunch


In [ ]:
# CELL 7 — CREATE MCP SERVER

server_code = '''
from mcp.server.fastmcp import FastMCP
import json
import os

mcp = FastMCP("Expense Tracker MCP Server")

DATA_FILE = "expenses.json"


def load_expenses():
    if not os.path.exists(DATA_FILE):
        return []

    with open(DATA_FILE, "r") as f:
        return json.load(f)


def save_expenses(expenses):
    with open(DATA_FILE, "w") as f:
        json.dump(expenses, f, indent=2)


@mcp.tool()
def add_expense(
    amount: float,
    category: str,
    description: str = "N/A"
) -> str:
    """Add an expense to the expense tracker."""

    expenses = load_expenses()

    expense = {
        "amount": amount,
        "category": category,
        "description": description
    }

    expenses.append(expense)
    save_expenses(expenses)

    return (
        f"Expense added successfully: "
        f"₹{amount} | {category} | {description}"
    )


@mcp.tool()
def get_expenses() -> str:
    """Get all recorded expenses."""

    expenses = load_expenses()

    if not expenses:
        return "No expenses found."

    return "\\n".join(
        f"₹{e['amount']} | "
        f"{e['category']} | "
        f"{e['description']}"
        for e in expenses
    )


if __name__ == "__main__":
    mcp.run(transport="streamable-http")
'''

with open("mcp_server.py", "w") as f:
    f.write(server_code)

print("✅ mcp_server.py created successfully!")

✅ mcp_server.py created successfully!


In [ ]:
# CELL 8 — SAVE EXPENSE DATA

import json

with open("expenses.json", "w") as f:
    json.dump(expenses, f, indent=2)

print("✅ Expense data saved!")
print(expenses)

✅ Expense data saved!
[{'amount': 200, 'category': 'Travel', 'description': 'Bus fare'}, {'amount': 150.0, 'category': 'Food', 'description': 'Lunch'}]


In [ ]:
# CELL 9 — START MCP SERVER

import subprocess
import sys
import time

mcp_process = subprocess.Popen(
    [sys.executable, "mcp_server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

time.sleep(3)

if mcp_process.poll() is None:
    print("✅ MCP server is running!")
    print("Server URL: http://127.0.0.1:8000/mcp")
else:
    output, error = mcp_process.communicate()
    print("❌ MCP server failed to start")
    print(error)

✅ MCP server is running!
Server URL: http://127.0.0.1:8000/mcp


In [ ]:
# CELL 10 — MCP CLIENT CONNECTION

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

MCP_URL = "http://127.0.0.1:8000/mcp"

async def connect_to_mcp():
    async with streamablehttp_client(MCP_URL) as (
        read_stream,
        write_stream,
        get_session_id
    ):
        async with ClientSession(
            read_stream,
            write_stream
        ) as session:

            await session.initialize()

            print("✅ Connected to MCP server!")

await connect_to_mcp()

✅ Connected to MCP server!


In [ ]:
# CELL 11 — LIST MCP TOOLS

async def list_mcp_tools():

    async with streamablehttp_client(MCP_URL) as (
        read_stream,
        write_stream,
        get_session_id
    ):
        async with ClientSession(
            read_stream,
            write_stream
        ) as session:

            await session.initialize()

            result = await session.list_tools()

            print("🔧 MCP TOOLS AVAILABLE:\n")

            for tool in result.tools:
                print("•", tool.name)
                print(" ", tool.description)
                print()

await list_mcp_tools()

🔧 MCP TOOLS AVAILABLE:

• add_expense
  Add an expense to the expense tracker.

• get_expenses
  Get all recorded expenses.



In [ ]:
# CELL 12 — TEST GET EXPENSES THROUGH MCP

async def test_get_expenses():

    async with streamablehttp_client(MCP_URL) as (
        read_stream,
        write_stream,
        get_session_id
    ):
        async with ClientSession(
            read_stream,
            write_stream
        ) as session:

            await session.initialize()

            result = await session.call_tool(
                "get_expenses",
                arguments={}
            )

            print("📋 EXPENSES FROM MCP SERVER:")
            print(result.content[0].text)

await test_get_expenses()

📋 EXPENSES FROM MCP SERVER:
₹200 | Travel | Bus fare
₹150.0 | Food | Lunch


In [ ]:
async def test_add_expense():
    async with streamablehttp_client(MCP_URL) as (
        read_stream,
        write_stream,
        get_session_id
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()

            result = await session.call_tool(
                "add_expense",
                arguments={
                    "amount": 300,
                    "category": "Food",
                    "description": "Dinner"
                }
            )

            print("➕ MCP ADD EXPENSE RESULT:")
            print(result.content[0].text)


await test_add_expense()

➕ MCP ADD EXPENSE RESULT:
Expense added successfully: ₹300.0 | Food | Dinner


In [ ]:
# CELL 14 — CONNECT MCP WITH LANGGRAPH

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class ExpenseState(TypedDict):
    question: str
    result: str
    answer: str


async def call_mcp_tool(tool_name, arguments):

    async with streamablehttp_client(MCP_URL) as (
        read_stream,
        write_stream,
        get_session_id
    ):
        async with ClientSession(
            read_stream,
            write_stream
        ) as session:

            await session.initialize()

            result = await session.call_tool(
                tool_name,
                arguments=arguments
            )

            return result.content[0].text


async def expense_node(state: ExpenseState):

    question = state["question"].lower()

    if "show" in question or "list" in question or "view" in question:
        result = await call_mcp_tool(
            "get_expenses",
            {}
        )

    else:
        result = "Please specify an expense to add."

    return {
        "question": state["question"],
        "result": result,
        "answer": result
    }


builder = StateGraph(ExpenseState)

builder.add_node("expense", expense_node)

builder.add_edge(START, "expense")
builder.add_edge("expense", END)

expense_graph = builder.compile()

print("✅ MCP + LangGraph connected!")

✅ MCP + LangGraph connected!


In [ ]:
# CELL 15 — TEST LANGGRAPH

result = await expense_graph.ainvoke({
    "question": "Show my expenses",
    "result": "",
    "answer": ""
})

print("🤖 Bot:")
print(result["answer"])

🤖 Bot:
₹200 | Travel | Bus fare
₹150.0 | Food | Lunch
₹300.0 | Food | Dinner


In [ ]:
# CELL 16 — FINAL EXPENSE TRACKER CHATBOT

print("=" * 60)
print("        MCP + LANGGRAPH EXPENSE TRACKER")
print("=" * 60)
print("Type 'exit' to stop.")

while True:

    question = input("\nYou: ")

    if question.lower() in ["exit", "quit", "bye"]:
        print("Bot: Goodbye! 👋")
        break

    result = await expense_graph.ainvoke({
        "question": question,
        "result": "",
        "answer": ""
    })

    print("\nBot:", result["answer"])

        MCP + LANGGRAPH EXPENSE TRACKER
Type 'exit' to stop.

You: Show my expenses

Bot: ₹200 | Travel | Bus fare
₹150.0 | Food | Lunch
₹300.0 | Food | Dinner

You: exit
Bot: Goodbye! 👋
